In [54]:
import pandas as pd
from PIL import Image
import numpy as np

import tensorflow as tf
from tensorflow.keras import datasets, layers, models

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

In [55]:
# Extract the sample paths and labels
csv = pd.read_csv('../data/english.csv')
paths = csv['image'].tolist()
y = csv['label'].tolist()

# Get the 62 classes
classes = csv['label'].unique().tolist()
classes

['0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 'A',
 'B',
 'C',
 'D',
 'E',
 'F',
 'G',
 'H',
 'I',
 'J',
 'K',
 'L',
 'M',
 'N',
 'O',
 'P',
 'Q',
 'R',
 'S',
 'T',
 'U',
 'V',
 'W',
 'X',
 'Y',
 'Z',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [56]:
# Convert class labels to integer
conversion = {}

cur = 0
for label in classes:
    conversion[label] = cur
    cur += 1

for i in range(len(y)):
    y[i] = conversion[f"{y[i]}"]
y

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,


In [57]:
# The images are originally 1200x900, so we'll resize to 64x64 and convert to grayscale
def preprocess_img(path):
    img = Image.open(f"../data/{path}")

    img = img.resize((64, 64))
    img = img.convert("L")

    img_arr = np.array(img)
    img_arr = img_arr / 255.0

    img_arr = np.expand_dims(img_arr, axis=-1)

    return img_arr

In [58]:
# Image preprocessing
X = []
for image in paths:
    X.append(preprocess_img(image))

# Stack 
X = np.array(X, dtype="float32")
y = np.array(y, dtype="int32")

In [59]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [60]:
# Try 1NN first
X_flat = X.reshape(X.shape[0], -1)

X_flat_train, X_flat_test, y_train, y_test = train_test_split(X_flat, y, test_size=0.2, random_state=42)

def one_nn(X_flat_train, y_train, X_flat_test):
    y_pred = []
    for test in X_flat_test:
        dist = np.sum((X_flat_train - test) ** 2, axis=1)
        y_pred.append(y_train[np.argmin(dist)])
    
    return np.array(y_pred)

y_pred = one_nn(X_flat_train, y_train, X_flat_test)

# Evaluate
print(classification_report(y_test, y_pred, target_names=classes))

              precision    recall  f1-score   support

           0       0.33      0.43      0.38        14
           1       0.17      0.27      0.21        11
           2       0.62      0.50      0.56        10
           3       0.56      0.25      0.34        20
           4       0.45      0.36      0.40        14
           5       0.36      0.31      0.33        16
           6       0.20      0.22      0.21         9
           7       0.47      0.60      0.53        15
           8       0.42      0.42      0.42        12
           9       0.57      0.73      0.64        11
           A       0.86      0.75      0.80        16
           B       0.38      0.33      0.35         9
           C       0.30      0.67      0.41         9
           D       0.83      0.50      0.62        10
           E       0.64      0.47      0.54        15
           F       0.64      0.50      0.56        14
           G       0.67      0.60      0.63        10
           H       0.62    

In [61]:
# Create CNN
def init_model():
    model = models.Sequential()

    model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 1)))
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Conv2D(128, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(62, activation="softmax"))

    return model

In [62]:
model = init_model()
model.summary()

C:\Users\Tiger\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 62, 62, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 62)             │         7,998 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 690,622 (2.63 MB)

 Trainable params: 690,622 (2.63 MB)

 Non-trainable params: 0 (0.00 B)

In [63]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=10,
                    validation_data=(X_test, y_test))

Epoch 1/10


C:\Users\Tiger\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\backend\tensorflow\nn.py:1216: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


86/86 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.0315 - loss: 4.0763 - val_accuracy: 0.0968 - val_loss: 3.7657
Epoch 2/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.2078 - loss: 3.1372 - val_accuracy: 0.4472 - val_loss: 2.2255
Epoch 3/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4468 - loss: 2.0216 - val_accuracy: 0.5982 - val_loss: 1.5352
Epoch 4/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5674 - loss: 1.5098 - val_accuracy: 0.6334 - val_loss: 1.3188
Epoch 5/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.6642 - loss: 1.1336 - val_accuracy: 0.6906 - val_loss: 1.1257
Epoch 6/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7225 - loss: 0.9051 - val_accuracy: 0.6950 - val_loss: 1.0977
Epoch 7/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7808 - loss: 0.7193 - val_accuracy: 0.7258 - val_loss: 1.0194
Epoch 8/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8032 - loss: 0.6147 - val_accuracy: 0.7331 - val_loss: 0.

In [64]:
# Evaluation
print(model.evaluate(X_test, y_test))

y_prob = model.predict(X_test)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(y_test, y_pred, target_names=classes))

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7405 - loss: 0.9985
[0.9984837770462036, 0.740469217300415]
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
              precision    recall  f1-score   support

           0       0.50      0.71      0.59        14
           1       0.38      0.27      0.32        11
           2       0.73      0.80      0.76        10
           3       0.84      0.80      0.82        20
           4       0.78      1.00      0.88        14
           5       1.00      0.94      0.97        16
           6       0.83      0.56      0.67         9
           7       0.92      0.80      0.86        15
           8       0.53      0.83      0.65        12
           9       0.70      0.64      0.67        11
           A       0.86      0.75      0.80        16
           B       0.86      0.67      0.75         9
           C       0.55      0.67      0.60         9
           D       0.71      1.00      0.83        10
           E       1.00      0.80    